In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Semilla para que los resultados sean reproducibles si lo ejecutas varias veces
np.random.seed(42)

# ==========================================
# 1. CONFIGURACIÓN Y PARÁMETROS BASE
# ==========================================
NUM_RESERVAS = 200000  # Cantidad dentro del rango (100k - 300k)

# Hoteles y ponderación según capacidad (310 + 350 + 270 = 930 habitaciones)
hoteles = ["Atlantico", "Jacaranda", "Tirma"]
prob_hoteles = [310 / 930, 350 / 930, 270 / 930]

# Fechas límite de check-in
fecha_inicio_checkin = datetime(2021, 1, 1)
fecha_fin_checkin = datetime(2026, 6, 30)
dias_totales_rango = (fecha_fin_checkin - fecha_inicio_checkin).days

# Dimensiones
booking_sources = ["Direct", "B2B", "Web"]
prob_sources = [0.20, 0.45, 0.35]

packages = ["Desayuno", "Media Pension", "All Inclusive"]
prob_packages = [0.25, 0.45, 0.30]

room_categories = ["Standard", "Superior", "Junior Suite", "Suite", "Deluxe Ocean View"]
prob_rooms = [0.45, 0.30, 0.12, 0.05, 0.08]

# Países emisores típicos de Gran Canaria
countries = ["España", "Reino Unido", "Alemania", "Suecia", "Noruega", "Holanda", "Francia", "Bélgica", "Irlanda", "Otros"]
prob_countries = [0.30, 0.25, 0.20, 0.06, 0.05, 0.05, 0.04, 0.02, 0.02, 0.01]

guest_types = ["Nuevo", "Repetidor", "Agente de Viaje", "Personal Compañia"]
prob_guest_types = [0.65, 0.28, 0.04, 0.03]

age_groups = ["18-25", "26-35", "36-50", "51-65", "65+"]
prob_age_groups = [0.10, 0.22, 0.33, 0.23, 0.12]

key_accounts = [
    "Booking.com", "TUI Group", "Jet2holidays", "Schauinsland-Reisen", 
    "Expedia", "Hotelbeds", "Viajes El Corte Inglés", "Direct Web / Call Center", 
    "Der Touristik", "Alltours"
]
prob_accounts = [0.22, 0.20, 0.15, 0.10, 0.08, 0.08, 0.06, 0.05, 0.03, 0.03]

# ==========================================
# 2. GENERACIÓN ALEATORIA VECTORIZADA
# ==========================================
print("Generando datos...")

# Generar RES_ID
res_ids = [f"RES-{i:07d}" for i in range(1, NUM_RESERVAS + 1)]

# Asignación de Hotel
hotel_col = np.random.choice(hoteles, size=NUM_RESERVAS, p=prob_hoteles)

# Generar Booking Date (Check-in)
dias_offset = np.random.randint(0, dias_totales_rango + 1, size=NUM_RESERVAS)
booking_dates = [fecha_inicio_checkin + timedelta(days=int(d)) for d in dias_offset]

# Lead time (Días de anticipación de la reserva) -> Distribución log-normal realista
lead_times = np.random.lognormal(mean=3.8, sigma=0.8, size=NUM_RESERVAS).astype(int)
lead_times = np.clip(lead_times, 0, 365) # Entre 0 y 365 días

# Booking Sale Date (Fecha en que se compró la reserva)
booking_sale_dates = [b_date - timedelta(days=int(lt)) for b_date, lt in zip(booking_dates, lead_times)]

# FIT vs GROUP
is_group = np.random.choice(["FIT", "GROUP"], size=NUM_RESERVAS, p=[0.88, 0.12])

# Group Type
group_types_list = ["GROUP", "Charter", "FAMTRIP", "FAM TIR"]
prob_group_types = [0.60, 0.25, 0.10, 0.05]
group_type_col = [
    np.random.choice(group_types_list, p=prob_group_types) if g == "GROUP" else "FIT / N/A" 
    for g in is_group
]

# Habitaciones y Noches
rooms_col = [
    np.random.randint(5, 25) if g == "GROUP" else np.random.choice([1, 2, 3], p=[0.85, 0.12, 0.03]) 
    for g in is_group
]

nights_col = np.random.choice(
    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14], 
    size=NUM_RESERVAS, 
    p=[0.05, 0.08, 0.10, 0.10, 0.08, 0.07, 0.30, 0.05, 0.03, 0.04, 0.02, 0.02, 0.01, 0.05]
)

# Pax (Pasajeros)
pax_col = [
    r * np.random.randint(2, 4) if g == "GROUP" else r * np.random.choice([1, 2, 3, 4], p=[0.20, 0.60, 0.12, 0.08])
    for r, g in zip(rooms_col, is_group)
]

# Selección de categorías y fuentes
source_col = np.random.choice(booking_sources, size=NUM_RESERVAS, p=prob_sources)
package_col = np.random.choice(packages, size=NUM_RESERVAS, p=prob_packages)
room_col = np.random.choice(room_categories, size=NUM_RESERVAS, p=prob_rooms)
country_col = np.random.choice(countries, size=NUM_RESERVAS, p=prob_countries)
guest_type_col = np.random.choice(guest_types, size=NUM_RESERVAS, p=prob_guest_types)
age_group_col = np.random.choice(age_groups, size=NUM_RESERVAS, p=prob_age_groups)
account_col = np.random.choice(key_accounts, size=NUM_RESERVAS, p=prob_accounts)

# ==========================================
# 3. CÁLCULO DE REVENUE NETO REALISTA
# ==========================================
# Precios base diarios por habitación según categoría
base_prices_room = {
    "Standard": 85,
    "Superior": 110,
    "Junior Suite": 150,
    "Suite": 210,
    "Deluxe Ocean View": 175
}

# Suplemento diario por tipo de paquete (por pax)
package_supplements = {
    "Desayuno": 12,
    "Media Pension": 28,
    "All Inclusive": 55
}

net_revenue = []
for i in range(NUM_RESERVAS):
    # Precio base habitación
    p_room = base_prices_room[room_col[i]]
    # Suplemento paquete por número de pax
    p_pkg = package_supplements[package_col[i]] * pax_col[i]
    
    # Factor estacional (Temporada Alta en Canarias: Noviembre a Abril + Julio/Agosto)
    mes = booking_dates[i].month
    factor_temporada = 1.25 if mes in [11, 12, 1, 2, 3, 4, 7, 8] else 0.95
    
    # Descuento según tipo de cliente
    factor_descuento = 1.0
    if guest_type_col[i] == "Personal Compañia":
        factor_descuento = 0.50
    elif guest_type_col[i] == "Agente de Viaje":
        factor_descuento = 0.70
    elif guest_type_col[i] == "Repetidor":
        factor_descuento = 0.90

    # Tarifa de grupo tiene un ligero descuento por volumen
    if is_group[i] == "GROUP":
        factor_descuento *= 0.85

    # Variación aleatoria diaria (+/- 10%)
    variacion = np.random.uniform(0.90, 1.10)

    # Cálculo Total Neto = (Habitación + Paquetes) * Noches * Multiplicadores
    rev_total = (p_room * rooms_col[i] + p_pkg) * nights_col[i] * factor_temporada * factor_descuento * variacion
    net_revenue.append(round(rev_total, 2))

# ==========================================
# 4. CONSTRUCCIÓN DEL DATAFRAME
# ==========================================
df_hotel = pd.DataFrame({
    'RES_ID': res_ids,
    'Hotel': hotel_col,
    
    # Fechas de Check-in (Booking Date)
    'Booking_Date': booking_dates,
    'Checkin_Day': [d.day for d in booking_dates],
    'Checkin_Month': [d.month for d in booking_dates],
    'Checkin_Month_Name': [d.strftime('%B') for d in booking_dates],
    'Checkin_Year': [d.year for d in booking_dates],
    'Checkin_Quarter': [f"Q{(d.month-1)//3 + 1}" for d in booking_dates],
    'Checkin_Day_of_Week': [d.strftime('%A') for d in booking_dates],
    
    # Fechas de Venta (Booking Sale Date)
    'Booking_Sale_Date': booking_sale_dates,
    'Sale_Day': [d.day for d in booking_sale_dates],
    'Sale_Month': [d.month for d in booking_sale_dates],
    'Sale_Year': [d.year for d in booking_sale_dates],
    'Lead_Time_Days': lead_times,
    
    # Métricas Operativas y Financieras
    'Rooms': rooms_col,
    'Pax': pax_col,
    'Nights': nights_col,
    'Net_Revenue_EUR': net_revenue,
    
    # Segmentos y Canales
    'Booking_Source': source_col,
    'Key_Account': account_col,
    'Fit_vs_Group': is_group,
    'Group_Type': group_type_col,
    'Package': package_col,
    'Room_Category': room_col,
    
    # Perfil del Cliente
    'Guest_Country': country_col,
    'Guest_Type': guest_type_col,
    'Age_Group': age_group_col
})

# ==========================================
# 5. GUARDAR Y REVISAR EL RESULTADO
# ==========================================
print(f"✅ ¡Dataset generado exitosamente con {len(df_hotel):,} filas!")

# Mostrar las primeras filas
print(df_hotel.head())

# Guardar en formato CSV o Parquet para cargar a tu herramienta de BI
ruta_salida_csv = r"C:\Users\andy_\Documents\Datasets\hoteles_gran_canaria\dataset_hoteles_gran_canaria.csv"
df_hotel.to_csv(ruta_salida_csv, index=False)
print(f"📁 Archivo guardado como: '{ruta_salida_csv}'")

Generando datos...
✅ ¡Dataset generado exitosamente con 200,000 filas!
        RES_ID      Hotel Booking_Date  Checkin_Day  Checkin_Month  \
0  RES-0000001  Jacaranda   2026-02-07            7              2   
1  RES-0000002      Tirma   2022-07-07            7              7   
2  RES-0000003      Tirma   2023-12-14           14             12   
3  RES-0000004  Jacaranda   2021-01-10           10              1   
4  RES-0000005  Atlantico   2026-06-18           18              6   

  Checkin_Month_Name  Checkin_Year Checkin_Quarter Checkin_Day_of_Week  \
0           February          2026              Q1            Saturday   
1               July          2022              Q3            Thursday   
2           December          2023              Q4            Thursday   
3            January          2021              Q1              Sunday   
4               June          2026              Q2            Thursday   

  Booking_Sale_Date  ...  Net_Revenue_EUR  Booking_Source    Ke

In [4]:
df = pd.read_csv(r"C:\Users\andy_\Documents\Datasets\hoteles_gran_canaria\dataset_hoteles_gran_canaria.csv")
df

,RES_ID,Hotel,Booking_Date,Checkin_Day,Checkin_Month,Checkin_Month_Name,Checkin_Year,Checkin_Quarter,Checkin_Day_of_Week,Booking_Sale_Date,...,Net_Revenue_EUR,Booking_Source,Key_Account,Fit_vs_Group,Group_Type,Package,Room_Category,Guest_Country,Guest_Type,Age_Group
0,RES-0000001,Jacaranda,2026-02-07,7,2,February,2026,Q1,Saturday,2025-12-18,...,1081.93,Web,Booking.com,FIT,FIT / N/A,Media Pension,Standard,Noruega,Nuevo,36-50
1,RES-0000002,Tirma,2022-07-07,7,7,July,2022,Q3,Thursday,2022-05-09,...,3619.17,B2B,Der Touristik,FIT,FIT / N/A,All Inclusive,Superior,Reino Unido,Nuevo,51-65
2,RES-0000003,Tirma,2023-12-14,14,12,December,2023,Q4,Thursday,2023-09-17,...,3228.68,Direct,Booking.com,GROUP,GROUP,Desayuno,Superior,Bélgica,Nuevo,36-50
3,RES-0000004,Jacaranda,2021-01-10,10,1,January,2021,Q1,Sunday,2020-10-06,...,1774.53,B2B,Booking.com,FIT,FIT / N/A,Media Pension,Deluxe Ocean View,Reino Unido,Repetidor,26-35
4,RES-0000005,Atlantico,2026-06-18,18,6,June,2026,Q2,Thursday,2026-01-12,...,25881.72,Web,TUI Group,GROUP,FAMTRIP,Media Pension,Deluxe Ocean View,España,Nuevo,26-35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,RES-0199996,Jacaranda,2026-01-22,22,1,January,2026,Q1,Thursday,2025-12-21,...,1589.77,Web,Booking.com,FIT,FIT / N/A,All Inclusive,Standard,España,Personal Compañia,36-50
199996,RES-0199997,Jacaranda,2021-08-07,7,8,August,2021,Q3,Saturday,2021-07-27,...,2486.26,B2B,Jet2holidays,FIT,FIT / N/A,Desayuno,Junior Suite,Reino Unido,Repetidor,26-35
199997,RES-0199998,Atlantico,2023-07-02,2,7,July,2023,Q3,Sunday,2023-06-02,...,2220.60,Web,Schauinsland-Reisen,FIT,FIT / N/A,Media Pension,Superior,España,Agente de Viaje,26-35
199998,RES-0199999,Jacaranda,2023-09-14,14,9,September,2023,Q3,Thursday,2023-07-27,...,504.70,Web,TUI Group,FIT,FIT / N/A,Media Pension,Standard,España,Nuevo,36-50


In [5]:
# 1. Crear las columnas base de Room Nights y Pax Nights
df['Room_Nights'] = df['Rooms'] * df['Nights']
df['Pax_Nights'] = df['Pax'] * df['Nights']

# 2. Ejemplo de Resumen / Cuadro de Mando agrupado por Hotel
resumen_hoteles = df.groupby('Hotel').agg(
    Total_Reservas=('RES_ID', 'count'),
    Total_Revenue=('Net_Revenue_EUR', 'sum'),
    Total_Room_Nights=('Room_Nights', 'sum'),
    Total_Pax_Nights=('Pax_Nights', 'sum'),
    Avg_LOS=('Nights', 'mean')
).reset_index()

# 3. Calcular métricas derivadas (ADR y RevPPN)
resumen_hoteles['ADR'] = resumen_hoteles['Total_Revenue'] / resumen_hoteles['Total_Room_Nights']
resumen_hoteles['RevPPN'] = resumen_hoteles['Total_Revenue'] / resumen_hoteles['Total_Pax_Nights']

print(resumen_hoteles[['Hotel', 'Total_Revenue', 'Avg_LOS', 'ADR', 'RevPPN']])

       Hotel  Total_Revenue   Avg_LOS         ADR     RevPPN
0  Atlantico   2.124073e+08  6.184107  185.722029  79.103625
1  Jacaranda   2.396842e+08  6.206513  185.850994  79.392293
2      Tirma   1.831124e+08  6.180098  184.902853  78.991956


In [6]:
df

,RES_ID,Hotel,Booking_Date,Checkin_Day,Checkin_Month,Checkin_Month_Name,Checkin_Year,Checkin_Quarter,Checkin_Day_of_Week,Booking_Sale_Date,...,Key_Account,Fit_vs_Group,Group_Type,Package,Room_Category,Guest_Country,Guest_Type,Age_Group,Room_Nights,Pax_Nights
0,RES-0000001,Jacaranda,2026-02-07,7,2,February,2026,Q1,Saturday,2025-12-18,...,Booking.com,FIT,FIT / N/A,Media Pension,Standard,Noruega,Nuevo,36-50,8,8
1,RES-0000002,Tirma,2022-07-07,7,7,July,2022,Q3,Thursday,2022-05-09,...,Der Touristik,FIT,FIT / N/A,All Inclusive,Superior,Reino Unido,Nuevo,51-65,14,28
2,RES-0000003,Tirma,2023-12-14,14,12,December,2023,Q4,Thursday,2023-09-17,...,Booking.com,GROUP,GROUP,Desayuno,Superior,Bélgica,Nuevo,36-50,20,60
3,RES-0000004,Jacaranda,2021-01-10,10,1,January,2021,Q1,Sunday,2020-10-06,...,Booking.com,FIT,FIT / N/A,Media Pension,Deluxe Ocean View,Reino Unido,Repetidor,26-35,7,14
4,RES-0000005,Atlantico,2026-06-18,18,6,June,2026,Q2,Thursday,2026-01-12,...,TUI Group,GROUP,FAMTRIP,Media Pension,Deluxe Ocean View,España,Nuevo,26-35,114,342
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,RES-0199996,Jacaranda,2026-01-22,22,1,January,2026,Q1,Thursday,2025-12-21,...,Booking.com,FIT,FIT / N/A,All Inclusive,Standard,España,Personal Compañia,36-50,12,24
199996,RES-0199997,Jacaranda,2021-08-07,7,8,August,2021,Q3,Saturday,2021-07-27,...,Jet2holidays,FIT,FIT / N/A,Desayuno,Junior Suite,Reino Unido,Repetidor,26-35,14,28
199997,RES-0199998,Atlantico,2023-07-02,2,7,July,2023,Q3,Sunday,2023-06-02,...,Schauinsland-Reisen,FIT,FIT / N/A,Media Pension,Superior,España,Agente de Viaje,26-35,15,30
199998,RES-0199999,Jacaranda,2023-09-14,14,9,September,2023,Q3,Thursday,2023-07-27,...,TUI Group,FIT,FIT / N/A,Media Pension,Standard,España,Nuevo,36-50,4,8


In [7]:
import os

# 1. Definir la carpeta y la ruta completa del archivo
carpeta_destino = r"C:\Users\andy_\Documents\Datasets\hoteles_gran_canaria"
ruta_completa = os.path.join(carpeta_destino, "dataset_hoteles_gran_canaria_final.csv")

# 2. Crear la carpeta por si aún no existe en tu equipo
os.makedirs(carpeta_destino, exist_ok=True)

# 3. Guardar el DataFrame
df_hotel.to_csv(ruta_completa, index=False)

print(f"💾 ¡CSV guardado con éxito en:\n{ruta_completa}")

💾 ¡CSV guardado con éxito en:
C:\Users\andy_\Documents\Datasets\hoteles_gran_canaria\dataset_hoteles_gran_canaria_final.csv
